# Basic Simulation Walkthrough: Gas Swelling in Nuclear Fuel

This notebook provides a **step-by-step introduction** to running gas swelling simulations for nuclear fuel materials. It is designed for researchers new to rate theory models.

## 📚 Navigation

- **Prerequisites**: Basic Python knowledge
- **Background Reading**: 
  - [Rate Theory Fundamentals](../docs/tutorials/rate_theory_fundamentals.md) - Physics background (20 min read)
  - [30-Minute Quickstart](../docs/tutorials/30minute_quickstart.md) - Installation guide
- **Related Notebooks**:
  - [Parameter Sweep Studies](02_Parameter_Sweep_Studies.ipynb) - Systematic parameter exploration
  - [Gas Distribution Analysis](03_Gas_Distribution_Analysis.ipynb) - Track gas across reservoirs
  - [Experimental Data Comparison](04_Experimental_Data_Comparison.ipynb) - Validate against experiments
- **Reference**: [Parameter Reference](../docs/parameter_reference.md) | [Interpreting Results](../docs/guides/interpreting_results.md)

## Learning Objectives

By the end of this tutorial, you will:
- Understand what the gas swelling model simulates
- Learn how to set up and run a basic simulation
- Interpret the output variables and results
- Create visualizations to understand bubble evolution
- Modify parameters to explore different conditions

## What is Gas Swelling?

During nuclear fission:
1. **Gas atoms** (Xenon, Krypton) are produced as fission products
2. These gas atoms **diffuse** through the fuel material
3. Gas atoms **nucleate bubbles** or accumulate at existing bubbles
4. Bubbles **grow** by absorbing more gas and vacancies
5. The fuel **swells** (expands) as bubbles occupy more volume

This model uses **rate theory** to track these processes over time, solving a system of 10 coupled differential equations.

---

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from gas_swelling.models.modelrk23 import GasSwellingModel
from gas_swelling.params.parameters import create_default_parameters

# Configure plotting for better visuals
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Libraries imported successfully!")
print("\nReady to run gas swelling simulations.")

## 1. Understanding the Model Parameters

The model requires several parameters that describe the material and operating conditions. Let's examine the default parameters.

In [ ]:
# Create default parameters
params = create_default_parameters()

# Display key parameters with physical meanings
print("=" * 70)
print("DEFAULT MODEL PARAMETERS")
print("=" * 70)
print(f"\n🌡️  Temperature: {params['temperature']} K")
print("   → Operating temperature of the fuel")
print("   → Typical range: 600-1000 K for metallic fuel")

print(f"\n⚛️  Fission rate: {params['fission_rate']:.2e} fissions/(m³·s)")
print("   → Rate of fission reactions producing gas atoms")
print("   → Higher rate = more gas production = faster swelling")

print(f"\n🔬 Dislocation density: {params['dislocation_density']:.2e} m⁻²")
print("   → Density of crystal defects (dislocations)")
print("   → Acts as sinks for vacancies and interstitials")
print("   → Affects defect concentrations and bubble growth")

print(f"\n💧 Surface energy: {params['surface_energy']} J/m²")
print("   → Energy cost of creating bubble surfaces")
print("   → Higher surface energy = smaller bubbles")
print("     (bubbles minimize surface area to reduce energy)")

print(f"\n🫧 Bulk nucleation factor (Fnb): {params['Fnb']:.2e}")
print(f"🫧 Boundary nucleation factor (Fnf): {params['Fnf']:.2e}")
print("   → Control the rate of new bubble formation")
print("   → Lower values = fewer bubbles, each bubble grows larger")
print("   → Higher values = more bubbles competing for gas")

print(f"\n📐 Gas equation of state: {params['eos_model']}")
print("   → 'ideal': Ideal gas law (PV = nRT)")
print("   → 'ronchi': Modified Van der Waals for high-pressure gas")

## 2. The 10 State Variables

The model tracks 10 coupled variables over time. Let's see what they are:

In [ ]:
# Create a model instance to see the state variables
model = GasSwellingModel(params)

print("=" * 70)
print("THE 10 STATE VARIABLES")
print("=" * 70)
print("\n📍 BULK (grain interior):")
print("   1. Cgb: Gas atom concentration in bulk matrix")
print("   2. Ccb: Bubble/cavity concentration in bulk")
print("   3. Ncb: Gas atoms per bulk bubble")
print("   4. cvb: Vacancy concentration in bulk")
print("   5. cib: Interstitial concentration in bulk")

print("\n📍 BOUNDARIES (grain boundaries/phase interfaces):")
print("   6. Cgf: Gas atom concentration at boundaries")
print("   7. Ccf: Cavity concentration at boundaries")
print("   8. Ncf: Gas atoms per boundary cavity")
print("   9. cvf: Vacancy concentration at boundaries")
print("  10. cif: Interstitial concentration at boundaries")

print(f"\n✓ Model initialized with {len(model.initial_state)} state variables")

## 3. Running Your First Simulation

Now let's run a simulation! We'll simulate 100 days of irradiation.

In [ ]:
# Simulation time: 100 days
sim_time_days = 100
sim_time_sec = sim_time_days * 24 * 3600

# Time points for output (solver adapts between these points)
t_eval = np.linspace(0, sim_time_sec, 100)

print(f"Running simulation for {sim_time_days} days at {params['temperature']} K...")
print("(This may take 10-30 seconds...)")

# Solve the system of ODEs
result = model.solve(
    t_span=(0, sim_time_sec),
    t_eval=t_eval
)

print("\n✅ Simulation completed successfully!")

## 4. Understanding the Results

The solver returns a dictionary with time series of all state variables plus derived quantities like bubble radii and swelling.

In [ ]:
# Convert time to days for easier interpretation
time_days = result['time'] / (24 * 3600)

# Calculate swelling over time
# Swelling = volume fraction occupied by bubbles
V_bubbles_bulk = (4/3) * np.pi * result['Rcb']**3 * result['Ccb']
V_bubbles_boundary = (4/3) * np.pi * result['Rcf']**3 * result['Ccf']
swelling = (V_bubbles_bulk + V_bubbles_boundary) * 100  # Convert to percent

print("=" * 70)
print("SIMULATION RESULTS (after {} days)".format(sim_time_days))
print("=" * 70)

print("\n📏 Final bubble radii:")
print(f"   Bulk bubbles: {result['Rcb'][-1]*1e9:.2f} nm")
print(f"   Boundary bubbles: {result['Rcf'][-1]*1e9:.2f} nm")

print("\n🔢 Final bubble concentrations:")
print(f"   Bulk bubbles: {result['Ccb'][-1]:.2e} bubbles/m³")
print(f"   Boundary bubbles: {result['Ccf'][-1]:.2e} bubbles/m³")

print("\n💥 Final swelling: {:.4f}%".format(swelling[-1]))

print("\n⚗️  Gas atoms per bubble:")
print(f"   Bulk bubbles: {result['Ncb'][-1]:.0f} atoms/bubble")
print(f"   Boundary bubbles: {result['Ncf'][-1]:.0f} atoms/bubble")

print("\n💨 Gas released: {:.2e} atoms/m³".format(result['released_gas'][-1]))

## 5. Visualizing Bubble Evolution

Let's create plots to see how the bubbles evolve over time.

In [ ]:
# Create a multi-panel visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle(f'Gas Swelling Evolution at {params["temperature"]} K', 
             fontsize=14, fontweight='bold')

# Plot 1: Bubble radius evolution
axes[0, 0].plot(time_days, result['Rcb']*1e9, label='Bulk bubbles', linewidth=2)
axes[0, 0].plot(time_days, result['Rcf']*1e9, label='Boundary bubbles', linewidth=2)
axes[0, 0].set_xlabel('Time (days)')
axes[0, 0].set_ylabel('Bubble radius (nm)')
axes[0, 0].set_title('Bubble Radius Evolution')
axes[0, 0].legend()

# Plot 2: Swelling evolution
axes[0, 1].plot(time_days, swelling, color='red', linewidth=2)
axes[0, 1].set_xlabel('Time (days)')
axes[0, 1].set_ylabel('Swelling (%)')
axes[0, 1].set_title('Fuel Swelling')

# Plot 3: Bubble concentration
axes[0, 2].semilogy(time_days, result['Ccb'], label='Bulk', linewidth=2)
axes[0, 2].semilogy(time_days, result['Ccf'], label='Boundary', linewidth=2)
axes[0, 2].set_xlabel('Time (days)')
axes[0, 2].set_ylabel('Bubble concentration (m⁻³)')
axes[0, 2].set_title('Bubble Concentration')
axes[0, 2].legend()

# Plot 4: Gas atoms per bubble
axes[1, 0].plot(time_days, result['Ncb'], label='Bulk', linewidth=2)
axes[1, 0].plot(time_days, result['Ncf'], label='Boundary', linewidth=2)
axes[1, 0].set_xlabel('Time (days)')
axes[1, 0].set_ylabel('Gas atoms per bubble')
axes[1, 0].set_title('Gas Accumulation in Bubbles')
axes[1, 0].legend()

# Plot 5: Gas in solution
axes[1, 1].plot(time_days, result['Cgb'], label='Bulk matrix', linewidth=2)
axes[1, 1].plot(time_days, result['Cgf'], label='Boundaries', linewidth=2)
axes[1, 1].set_xlabel('Time (days)')
axes[1, 1].set_ylabel('Gas concentration (atoms/m³)')
axes[1, 1].set_title('Gas in Solution')
axes[1, 1].legend()

# Plot 6: Gas release
axes[1, 2].plot(time_days, result['released_gas'], linewidth=2, color='purple')
axes[1, 2].set_xlabel('Time (days)')
axes[1, 2].set_ylabel('Released gas (atoms/m³)')
axes[1, 2].set_title('Cumulative Gas Release')

plt.tight_layout()
plt.show()

print("✅ Plots generated successfully!")

## 6. Physical Interpretation of Results

Let's interpret what we're seeing in the plots.

In [ ]:
# Calculate key metrics for interpretation
initial_radius_bulk = result['Rcb'][0] * 1e9
final_radius_bulk = result['Rcb'][-1] * 1e9
initial_radius_boundary = result['Rcf'][0] * 1e9
final_radius_boundary = result['Rcf'][-1] * 1e9

print("=" * 70)
print("PHYSICAL INTERPRETATION")
print("=" * 70)

print("\n🔍 BUBBLE GROWTH:")
growth_factor_bulk = final_radius_bulk / initial_radius_bulk if initial_radius_bulk > 0 else 0
growth_factor_boundary = final_radius_boundary / initial_radius_boundary if initial_radius_boundary > 0 else 0
print(f"   Bulk bubbles grew from {initial_radius_bulk:.3f} nm to {final_radius_bulk:.2f} nm")
if growth_factor_bulk > 0:
    print(f"   → Growth factor: {growth_factor_bulk:.1f}x")
print(f"   Boundary bubbles grew from {initial_radius_boundary:.3f} nm to {final_radius_boundary:.2f} nm")
if growth_factor_boundary > 0:
    print(f"   → Growth factor: {growth_factor_boundary:.1f}x")

print("\n🔍 SWELLING MECHANISM:")
print("   Swelling depends on TWO competing factors:")
print(f"   1. Bubble radius (R): Grows from {initial_radius_bulk:.3f} → {final_radius_bulk:.2f} nm")
print(f"      → Volume ∝ R³, so even small R increases have large effects")
print(f"   2. Bubble concentration (Cc): {result['Ccb'][-1]:.2e} bubbles/m³")
print(f"   3. Swelling = R³ × Cc → {swelling[-1]:.4f}%")

print("\n🔍 GAS DISTRIBUTION:")
gas_in_bulk_matrix = result['Cgb'][-1]
gas_in_bulk_bubbles = result['Ccb'][-1] * result['Ncb'][-1]
gas_in_boundary_bubbles = result['Ccf'][-1] * result['Ncf'][-1]
gas_released = result['released_gas'][-1]
total_gas = gas_in_bulk_matrix + gas_in_bulk_bubbles + gas_in_boundary_bubbles + gas_released

if total_gas > 0:
    print(f"   In bulk matrix: {gas_in_bulk_matrix/total_gas*100:.1f}%")
    print(f"   In bulk bubbles: {gas_in_bulk_bubbles/total_gas*100:.1f}%")
    print(f"   In boundary bubbles: {gas_in_boundary_bubbles/total_gas*100:.1f}%")
    print(f"   Released: {gas_released/total_gas*100:.1f}%")

print("\n🔍 EVOLUTION STAGES:")
print("   1. Early time: Gas atoms diffuse, bubble nucleation begins")
print("   2. Intermediate: Bubbles grow by absorbing gas and vacancies")
print("   3. Late time: Bubbles may interconnect, gas release accelerates")

print("=" * 70)

## 7. Modifying Parameters: Temperature Effects

One of the most important parameters is temperature. Let's compare swelling at different temperatures.

In [ ]:
# Define temperatures to compare
temperatures = [600, 700, 800, 900]  # Kelvin

print("Running simulations at different temperatures...")
print("(This may take 30-60 seconds...)\n")

# Storage for results
temp_results = {}

for temp in temperatures:
    print(f"Running at {temp} K...", end=" ")
    
    # Create parameters with this temperature
    params_temp = create_default_parameters()
    params_temp['temperature'] = temp
    
    # Create model and solve
    model_temp = GasSwellingModel(params_temp)
    result_temp = model_temp.solve(
        t_span=(0, sim_time_sec),
        t_eval=t_eval
    )
    
    # Calculate swelling
    Vb = (4/3) * np.pi * result_temp['Rcb']**3 * result_temp['Ccb']
    Vf = (4/3) * np.pi * result_temp['Rcf']**3 * result_temp['Ccf']
    swelling_temp = (Vb + Vf) * 100
    
    # Store results
    temp_results[temp] = {
        'swelling': swelling_temp,
        'Rcb': result_temp['Rcb'],
        'Rcf': result_temp['Rcf']
    }
    
    print(f"✓ Final swelling: {swelling_temp[-1]:.4f}%")

print("\n✅ All simulations completed!")

In [ ]:
# Plot temperature comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Swelling evolution at different temperatures
colors = ['blue', 'green', 'orange', 'red']
for i, temp in enumerate(temperatures):
    axes[0].plot(time_days, temp_results[temp]['swelling'],
                linewidth=2, color=colors[i], label=f'{temp} K')

axes[0].set_xlabel('Time (days)', fontsize=11)
axes[0].set_ylabel('Swelling (%)', fontsize=11)
axes[0].set_title('Swelling Evolution at Different Temperatures', fontweight='bold')
axes[0].legend()

# Plot 2: Final swelling vs temperature
final_swelling = [temp_results[t]['swelling'][-1] for t in temperatures]
axes[1].plot(temperatures, final_swelling, 'o-', linewidth=2, markersize=10, color='red')
axes[1].set_xlabel('Temperature (K)', fontsize=11)
axes[1].set_ylabel('Final Swelling (%)', fontsize=11)
axes[1].set_title('Temperature Dependence of Swelling', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔍 Temperature Effect Interpretation:")
print("   - Low T (600 K): Slow diffusion limits gas transport to bubbles")
print("   - Medium T (700-800 K): Optimal balance for bubble growth")
print("   - High T (900 K): Thermal effects begin to reduce bubble stability")

## 8. Exploring Other Parameters

Let's experiment with changing the fission rate, which controls how fast gas is produced.

In [ ]:
# Test different fission rates
fission_rates = [1e19, 2e19, 4e19, 6e19]  # fissions/(m³·s)
test_temp = 750  # K (near peak swelling)

print(f"Testing fission rates at {test_temp} K...\n")

fr_results = []
for fr in fission_rates:
    # Create parameters
    p = create_default_parameters()
    p['temperature'] = test_temp
    p['fission_rate'] = fr
    
    # Run simulation
    m = GasSwellingModel(p)
    r = m.solve(t_span=(0, sim_time_sec), t_eval=t_eval)
    
    # Calculate swelling
    Vb = (4/3) * np.pi * r['Rcb']**3 * r['Ccb']
    Vf = (4/3) * np.pi * r['Rcf']**3 * r['Ccf']
    s = (Vb + Vf) * 100
    
    fr_results.append(s[-1])
    print(f"  Fission rate {fr:.1e}: Final swelling = {s[-1]:.4f}%")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.array(fission_rates)/1e19, fr_results, 'o-', linewidth=2, markersize=10)
ax.set_xlabel('Fission Rate (×10¹⁹ fissions/(m³·s))')
ax.set_ylabel('Final Swelling (%)')
ax.set_title(f'Effect of Fission Rate at {test_temp} K')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ Observation: Higher fission rate → more gas production → more swelling")

## 9. Quick Reference: Modifying Parameters

Here's a quick guide for modifying common parameters:

In [ ]:
print("=" * 70)
print("QUICK REFERENCE: MODIFYING PARAMETERS")
print("=" * 70)

print("\n📋 How to modify parameters:")
print("""
# Step 1: Create default parameters
params = create_default_parameters()

# Step 2: Modify specific parameters
params['temperature'] = 800              # Temperature (K)
params['fission_rate'] = 3e20           # Fission rate (fissions/m³/s)
params['dislocation_density'] = 5e13    # Dislocation density (m⁻²)
params['surface_energy'] = 0.6          # Surface energy (J/m²)
params['Fnb'] = 5e-6                   # Bulk nucleation factor
params['Fnf'] = 5e-6                   # Boundary nucleation factor
params['eos_model'] = 'ronchi'         # Gas EOS: 'ideal' or 'ronchi'

# Step 3: Create model with modified parameters
model = GasSwellingModel(params)

# Step 4: Run simulation
result = model.solve(t_span=(0, sim_time), t_eval=t_eval)
""")

print("\n📚 Parameter ranges:")
print("   Temperature: 600-1000 K (metallic fuel operating range)")
print("   Fission rate: 1e19 - 1e21 fissions/(m³·s)")
print("   Dislocation density: 1e13 - 1e15 m⁻²")
print("   Surface energy: 0.3 - 1.0 J/m²")
print("   Nucleation factors: 1e-7 - 1e-4")

print("=" * 70)

## 10. Summary and Next Steps

### What We Learned

1. **Model Basics**:
   - The model tracks 10 coupled variables describing gas bubble evolution
   - It simulates gas production, diffusion, bubble nucleation, and growth
   - Swelling is the volume fraction occupied by bubbles

2. **Running Simulations**:
   - Use `create_default_parameters()` to get a starting point
   - Modify parameters as needed for your study
   - Call `model.solve()` to run the simulation

3. **Understanding Results**:
   - Bubble radius (R) grows as gas accumulates
   - Bubble concentration (Cc) depends on nucleation
   - Swelling = R³ × Cc (volume fraction)
   - Temperature has a non-monotonic effect on swelling

4. **Parameter Effects**:
   - Higher temperature → faster diffusion → typically larger bubbles
   - Higher fission rate → more gas production → more swelling
   - Nucleation factors control bubble number vs size trade-off

### Next Steps

**🔬 Continue with More Notebooks:**

1. **[02_Parameter_Sweep_Studies.ipynb](02_Parameter_Sweep_Studies.ipynb)**
   - Systematic parameter variations
   - Temperature, fission rate, dislocation density sweeps
   - Two-parameter interaction studies
   - Sensitivity analysis basics

2. **[03_Gas_Distribution_Analysis.ipynb](03_Gas_Distribution_Analysis.ipynb)**
   - Detailed gas tracking across all reservoirs
   - Pie charts and stacked area plots
   - Bulk vs boundary gas partitioning
   - Gas release kinetics

3. **[04_Experimental_Data_Comparison.ipynb](04_Experimental_Data_Comparison.ipynb)**
   - Validate model against U-10Zr and U-Pu-Zr data
   - Understand model limitations
   - Quantitative validation metrics

4. **[05_Custom_Material_Composition.ipynb](05_Custom_Material_Composition.ipynb)**
   - U-Zr alloys (6-12% Zr)
   - U-Pu-Zr alloys (10-25% Pu)
   - Create custom compositions

5. **[06_Advanced_Analysis_Techniques.ipynb](06_Advanced_Analysis_Techniques.ipynb)**
   - Global sensitivity analysis (Morris, Sobol)
   - Monte Carlo uncertainty quantification
   - Parameter prioritization

**📖 Deepen Your Understanding:**

**Tutorials:**
- [Rate Theory Fundamentals](../docs/tutorials/rate_theory_fundamentals.md) - Physics background
- [Model Equations Explained](../docs/tutorials/model_equations_explained.md) - All 10 variables detailed
- [30-Minute Quickstart](../docs/tutorials/30minute_quickstart.md) - Installation and setup

**Guides:**
- [Interpreting Results Guide](../docs/guides/interpreting_results.md) - Understand all output variables
- [Plot Gallery](../docs/guides/plot_gallery.md) - Publication-quality visualizations
- [Troubleshooting Guide](../docs/guides/troubleshooting.md) - Solve common issues

**Reference:**
- [Parameter Reference](../docs/parameter_reference.md) - Complete parameter documentation

**💻 Example Scripts:**
- [examples/quickstart_tutorial.py](../examples/quickstart_tutorial.py) - Command-line version
- [examples/temperature_sweep_plotting.py](../examples/temperature_sweep_plotting.py) - Temperature studies
- [examples/plotting_examples.py](../examples/plotting_examples.py) - More visualization examples

### Further Exploration Ideas

- **Bubble size distribution**: How does nucleation factor affect bubble number vs size?
- **Time evolution**: How does the peak swelling temperature change over time?
- **Gas release**: Under what conditions does significant gas release occur?
- **Composition effects**: How do U-Zr vs U-Pu-Zr alloys differ?
- **Parameter sensitivity**: Which parameters most strongly affect swelling?

---

**🎉 Congratulations!** You've completed the basic simulation walkthrough.

You now have the skills to:
- ✅ Set up and run gas swelling simulations
- ✅ Interpret the output variables
- ✅ Modify parameters to explore different conditions
- ✅ Create visualizations of bubble evolution

**Ready for more?** Continue to [Parameter Sweep Studies](02_Parameter_Sweep_Studies.ipynb) to systematically explore parameter effects!

Happy simulating! 🚀